# 💼 New Hire Salary vs Inflation vs Loyalty Pay Pipeline
**Author:** Allen Ramirez &nbsp;|&nbsp; **Data Source:** FRED (Federal Reserve Economic Data) &nbsp;|&nbsp; **Refreshes:** Every run pulls live data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/webuild262-boop/salary-inflation-loyalty-pipeline/blob/main/salary_loyalty_pipeline.ipynb)

---
A 12-step ML pipeline that **quantifies the loyalty penalty** — the gap between what companies pay new hires (market wages) versus how much they raise existing employees (incumbent raises), both measured against inflation.

| Series | FRED ID | Frequency | What it represents |
|--------|---------|-----------|--------------------|
| CPI | `CPIAUCSL` | Monthly | Inflation (All Urban Consumers) |
| AHE | `AHETPI` | Monthly | Market wages / new hire pay proxy |
| ECI | `ECIWAG` | Quarterly | Incumbent employee raises |

**Loyalty Penalty = AHE growth − ECI growth.** When positive, new hires earn more in wage growth than loyal employees receive in raises.

In [ ]:
# ── Step 1 ── Install & Import Libraries ──────────────────────────────────────
!pip install -q pandas numpy scikit-learn matplotlib requests

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from io import StringIO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')

In [ ]:
# ── Step 2 ── Pull CPI (Inflation) from FRED ──────────────────────────────────
# CPIAUCSL: Consumer Price Index for All Urban Consumers — Monthly, Seasonally Adjusted
CPI_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL'

response = requests.get(CPI_URL, timeout=30)
df_cpi = pd.read_csv(StringIO(response.text), parse_dates=['DATE'])
df_cpi.columns = ['date', 'cpi']
df_cpi = df_cpi[df_cpi['date'] >= '2000-01-01'].copy()
df_cpi['date'] = pd.to_datetime(df_cpi['date'])

print(f'✅ CPI fetched: {len(df_cpi)} monthly rows  '
      f'({df_cpi["date"].min().date()} → {df_cpi["date"].max().date()})')
df_cpi.tail()

In [ ]:
# ── Step 3 ── Pull Average Hourly Earnings (Market / New Hire Wages) ──────────
# AHETPI: Average Hourly Earnings of All Employees, Total Private — Monthly, SA
AHE_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=AHETPI'

response = requests.get(AHE_URL, timeout=30)
df_ahe = pd.read_csv(StringIO(response.text), parse_dates=['DATE'])
df_ahe.columns = ['date', 'ahe']
df_ahe = df_ahe[df_ahe['date'] >= '2000-01-01'].copy()
df_ahe['date'] = pd.to_datetime(df_ahe['date'])

print(f'✅ AHE (market wages) fetched: {len(df_ahe)} monthly rows  '
      f'({df_ahe["date"].min().date()} → {df_ahe["date"].max().date()})')
df_ahe.tail()

In [ ]:
# ── Step 4 ── Pull Employment Cost Index (Incumbent Raises) ───────────────────
# ECIWAG: Employment Cost Index: Wages & Salaries, Private Industry — Quarterly, SA
ECI_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=ECIWAG'

response = requests.get(ECI_URL, timeout=30)
df_eci = pd.read_csv(StringIO(response.text), parse_dates=['DATE'])
df_eci.columns = ['date', 'eci']
df_eci = df_eci[df_eci['date'] >= '2000-01-01'].copy()
df_eci['date'] = pd.to_datetime(df_eci['date'])

print(f'✅ ECI (incumbent raises) fetched: {len(df_eci)} quarterly rows  '
      f'({df_eci["date"].min().date()} → {df_eci["date"].max().date()})')
df_eci.tail()

In [ ]:
# ── Step 5 ── Resample Monthly → Quarterly & Merge All Three Series ───────────
# Resample CPI and AHE to quarterly (last observation per quarter)
df_cpi_q = df_cpi.set_index('date').resample('QS').last().reset_index()
df_ahe_q = df_ahe.set_index('date').resample('QS').last().reset_index()

# Align ECI quarterly dates to quarter-start
df_eci_q = df_eci.copy()
df_eci_q['date'] = df_eci_q['date'].dt.to_period('Q').dt.to_timestamp()

# Merge on date
df = df_cpi_q.merge(df_ahe_q, on='date').merge(df_eci_q, on='date')
df = df.dropna().reset_index(drop=True)

print(f'✅ Merged dataset: {len(df)} quarterly rows')
print(f'   Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'   Columns    : {list(df.columns)}')
df.head()

In [ ]:
# ── Step 6 ── Year-Over-Year Growth Rates ─────────────────────────────────────
df['cpi_yoy'] = df['cpi'].pct_change(4) * 100   # Inflation rate
df['ahe_yoy'] = df['ahe'].pct_change(4) * 100   # Market wage growth
df['eci_yoy'] = df['eci'].pct_change(4) * 100   # Incumbent raise rate

df = df.dropna().reset_index(drop=True)

latest = df.iloc[-1]
print(f'✅ YoY growth rates calculated — {len(df)} rows remaining')
print(f'\nLatest quarter ({latest["date"].date()}):')
print(f'  Inflation (CPI YoY):       {latest["cpi_yoy"]:.2f}%')
print(f'  Market Wage Growth (AHE):  {latest["ahe_yoy"]:.2f}%')
print(f'  Incumbent Raises (ECI):    {latest["eci_yoy"]:.2f}%')
df.tail()

In [ ]:
# ── Step 7 ── Loyalty Penalty Index ──────────────────────────────────────────
# Loyalty Penalty = Market Wage Growth − Incumbent Raise Rate
# Positive → new hires gaining faster than loyal employees (penalty active)
# Negative → loyal employees keeping pace or ahead
df['loyalty_penalty'] = df['ahe_yoy'] - df['eci_yoy']
df['real_wage_growth'] = df['ahe_yoy'] - df['cpi_yoy']   # Inflation-adjusted
df['penalty_label']   = (df['loyalty_penalty'] > 0).astype(int)

penalty_quarters = df['penalty_label'].sum()
total_quarters   = len(df)
pct_penalty      = penalty_quarters / total_quarters * 100

print('✅ Loyalty Penalty Index created')
print(f'\n📊 Summary ({df["date"].min().year}–{df["date"].max().year}):')
print(f'   Quarters WITH penalty   : {penalty_quarters} / {total_quarters}  ({pct_penalty:.1f}%)')
print(f'   Latest penalty value    : {df["loyalty_penalty"].iloc[-1]:+.2f}%')
print(f'   Worst penalty on record : {df["loyalty_penalty"].max():+.2f}%')
print(f'   Avg real wage growth    : {df["real_wage_growth"].mean():.2f}%')

In [ ]:
# ── Step 8 ── Feature Engineering ────────────────────────────────────────────
df['cpi_lag1']       = df['cpi_yoy'].shift(1)
df['ahe_lag1']       = df['ahe_yoy'].shift(1)
df['eci_lag1']       = df['eci_yoy'].shift(1)
df['penalty_lag1']   = df['loyalty_penalty'].shift(1)
df['penalty_roll4']  = df['loyalty_penalty'].rolling(4).mean()
df['ahe_eci_spread'] = df['ahe_yoy'] - df['eci_yoy']
df['cpi_ahe_spread'] = df['cpi_yoy'] - df['ahe_yoy']

df = df.dropna().reset_index(drop=True)

FEATURES = [
    'cpi_yoy', 'ahe_yoy', 'eci_yoy',
    'cpi_lag1', 'ahe_lag1', 'eci_lag1',
    'penalty_lag1', 'penalty_roll4',
    'ahe_eci_spread', 'cpi_ahe_spread'
]

print(f'✅ Feature engineering complete — {len(df)} rows, {len(FEATURES)} features')
print(f'   Features: {FEATURES}')

In [ ]:
# ── Step 9 ── Chronological Train / Test Split (80 / 20) ─────────────────────
X = df[FEATURES]
y = df['penalty_label']

split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print('✅ Train / test split complete (chronological — no shuffle)')
print(f'   Training : {len(X_train)} rows  '
      f'({df["date"].iloc[0].date()} → {df["date"].iloc[split_idx-1].date()})')
print(f'   Test     : {len(X_test)} rows  '
      f'({df["date"].iloc[split_idx].date()} → {df["date"].iloc[-1].date()})')
print(f'   Penalty rate — Train: {y_train.mean()*100:.1f}%  |  Test: {y_test.mean()*100:.1f}%')

In [ ]:
# ── Step 10 ── Train Random Forest Classifier ─────────────────────────────────
# Target: predict whether the loyalty penalty will be active (1) or not (0)
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    class_weight='balanced'
)
rf.fit(X_train, y_train)

y_pred   = rf.predict(X_test)
accuracy = (y_pred == y_test).mean() * 100

print(f'✅ Random Forest trained — {rf.n_estimators} trees, max_depth={rf.max_depth}')
print(f'\n📊 Test Set Performance:')
print(f'   Accuracy: {accuracy:.1f}%')
print()
print(classification_report(y_test, y_pred, target_names=['No Penalty', 'Penalty Active']))

In [ ]:
# ── Step 11 ── Dashboard: Loyalty Penalty, Wage Trends, Model Results ─────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('New Hire Salary vs Inflation vs Loyalty Pay — Dashboard',
             fontsize=15, fontweight='bold', y=1.01)

# ── Chart 1: Loyalty Penalty over Time
ax1 = axes[0, 0]
ax1.plot(df['date'], df['loyalty_penalty'], color='#2563eb', linewidth=2)
ax1.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax1.fill_between(df['date'], df['loyalty_penalty'], 0,
                 where=(df['loyalty_penalty'] > 0), alpha=0.25,
                 color='#dc2626', label='Penalty Active (new hires ahead)')
ax1.fill_between(df['date'], df['loyalty_penalty'], 0,
                 where=(df['loyalty_penalty'] <= 0), alpha=0.25,
                 color='#16a34a', label='Employees Holding Ground')
ax1.set_title('Loyalty Penalty Index (2000–Present)', fontweight='bold')
ax1.set_ylabel('AHE Growth − ECI Growth (%)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# ── Chart 2: AHE vs ECI vs CPI
ax2 = axes[0, 1]
ax2.plot(df['date'], df['ahe_yoy'], label='Market Wages / AHE', color='#2563eb', linewidth=2)
ax2.plot(df['date'], df['eci_yoy'], label='Incumbent Raises / ECI', color='#16a34a', linewidth=2)
ax2.plot(df['date'], df['cpi_yoy'], label='Inflation / CPI', color='#dc2626',
         linewidth=2, linestyle='--')
ax2.set_title('Market Wages vs Incumbent Raises vs Inflation', fontweight='bold')
ax2.set_ylabel('YoY Growth (%)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# ── Chart 3: Confusion Matrix
ax3 = axes[1, 0]
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Penalty', 'Penalty']).plot(
    ax=ax3, colorbar=False, cmap='Blues')
ax3.set_title(f'Model Confusion Matrix  (Accuracy: {accuracy:.1f}%)', fontweight='bold')

# ── Chart 4: Feature Importance
ax4 = axes[1, 1]
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
colors = ['#2563eb' if i >= len(importances) - 3 else '#93c5fd'
          for i in range(len(importances))]
importances.plot(kind='barh', ax=ax4, color=colors)
ax4.set_title('Feature Importance (Top 3 highlighted)', fontweight='bold')
ax4.set_xlabel('Importance Score')
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('loyalty_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved — loyalty_dashboard.png')

In [ ]:
# ── Step 12 ── Forecast Next Quarter ─────────────────────────────────────────
latest_features  = df[FEATURES].iloc[[-1]]
pred_proba       = rf.predict_proba(latest_features)[0]
pred_label       = rf.predict(latest_features)[0]
current_penalty  = df['loyalty_penalty'].iloc[-1]
current_quarter  = df['date'].iloc[-1]
next_quarter     = current_quarter + pd.DateOffset(months=3)

verdict = '🔴 LOYALTY PENALTY ACTIVE' if pred_label == 1 else '🟢 EMPLOYEES HOLDING GROUND'

print('=' * 58)
print(f'  📅 NEXT-QUARTER FORECAST: {next_quarter.strftime("%B %Y")}')
print('=' * 58)
print(f'  Prediction  :  {verdict}')
print(f'  Confidence  :  {max(pred_proba)*100:.1f}%')
print(f'  Current loyalty penalty :  {current_penalty:+.2f}%')
print(f'  Current AHE growth      :  {df["ahe_yoy"].iloc[-1]:.2f}%')
print(f'  Current ECI growth      :  {df["eci_yoy"].iloc[-1]:.2f}%')
print(f'  Current inflation (CPI) :  {df["cpi_yoy"].iloc[-1]:.2f}%')
print('=' * 58)
if pred_label == 1:
    print(f'\n  Loyal employees are receiving {abs(current_penalty):.1f}% less in wage')
    print(f'  growth than the market is paying new hires — while also')
    print(f'  losing purchasing power to {df["cpi_yoy"].iloc[-1]:.1f}% inflation.')
else:
    print(f'\n  Incumbent raises are keeping pace with market wages.')
    print(f'  Loyalty penalty not currently active.')